In [1]:
import pandas as pd

In [2]:
df_features = pd.read_feather('df_features.fth')
df_labels = pd.read_feather('df_labels.fth')

In [3]:
df_labels
df_text_a = df_features[['index','headline']].rename(columns={'index':'headlinea','headline':'texta'})
df_text_b = df_features[['index','headline']].rename(columns={'index':'headlineb','headline':'textb'})

In [4]:
df_merged = df_labels.merge(df_text_a, on=['headlinea'],how='left').merge(df_text_b, on=['headlineb'],how='left')
df_merged['pairkey'] = df_merged.apply(lambda row: min(row['headlinea'],row['headlineb'])+max(row['headlinea'],row['headlineb']),axis=1)
df_merged['pairkey'].value_counts().head()

201MID252MID        2
36358NYT37299NYT    2
30505NYT37010NYT    2
24875NYT30148NYT    2
126MID127MID        1
Name: pairkey, dtype: int64

In [5]:
#There are some repeated pairs of events
df_merged_nodups= df_merged.drop_duplicates(subset=['pairkey'])

In [6]:
df_merged_nodups.coref.value_counts()

0.0    28661
1.0     5733
Name: coref, dtype: int64

In [7]:
## I have to specify again the criteria for matching the pairs taking into account the index

In [14]:
df_merged_nodups.loc[:, "coref"] = df_merged_nodups["coref"].astype(int)

/tmp/ipykernel_28974/1507682760.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merged_nodups.loc[:, "coref"] = df_merged_nodups["coref"].astype(int)


In [15]:
df_merged_nodups.drop(["index","headlinea","headlineb","pairkey"],axis=1).reset_index().to_feather("HoWMerge.fth")